# Split cleaned chess data

Run all cells after cleaning the sources. The notebook writes reproducible `train.txt`, `validation.txt`, `test.txt`, and `split_info.json` to `dataset/splits/`. Each split keeps approximately the same **class proportions** as the cleaned data; this does not make the classes equally frequent.

In [ ]:
from pathlib import Path
import sys

project_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'scripts' / 'split_dataset.py').is_file()
                     and (path / 'dataset').is_dir()), None)
if project_root is None:
    raise FileNotFoundError('Open this notebook from the MyTorch project or notebooks/ folder')
sys.path.insert(0, str(project_root))
from scripts.split_dataset import split_dataset

input_file = project_root / 'dataset' / 'processed' / 'cleaned.txt'
output_dir = project_root / 'dataset' / 'splits'
print('Input:', input_file)

## Settings

Change these values, then run all cells again. Balancing is off by default. When enabled, it repeats minority-class rows **in training only**; validation and test keep their original distribution.

In [ ]:
seed = 42
ratios = (0.8, 0.1, 0.1)  # train, validation, test
balance_training = False

In [ ]:
info = split_dataset(input_file, output_dir, seed=seed, ratios=ratios,
                     balance_training=balance_training)
print('Saved to:', output_dir)
print('Input SHA-256:', info['input_sha256'])

## Class counts

`Before` counts each original row once. `Output` includes any optional repetitions in training. The overlap check also runs inside the splitter before it writes files.

In [ ]:
print(f"{'Split':<12} {'Stage':<8} {'Nothing':>10} {'Check':>10} {'Checkmate':>10} {'Total':>10}")
for name in ('train', 'validation', 'test'):
    for stage, section in (('Before', 'before_balancing'), ('Output', 'output')):
        counts = info[section][name]
        print(f"{name:<12} {stage:<8} {counts['Nothing']:>10,} {counts['Check']:>10,} "
              f"{counts['Checkmate']:>10,} {sum(counts.values()):>10,}")
print('Overlapping model inputs:', info['overlapping_inputs'])